# PARTE 2 | Big Data — Pré-processamento e Análise com Polars

Nesta etapa, a base de dados expandida da fase anterior será utilizada para atender aos critérios mínimos de Big Data.

O processamento será realizado com a biblioteca **Polars**, utilizando os recursos para manipulação e análise de grandes volumes de dados.

# Contexto da Análise

## Pergunta principal
Esta análise busca responder à seguinte questão:

**Poços de petróleo mais profundos levam mais tempo para concluir uma intervenção de sonda?**

---

## Hipótese

A hipótese considerada é que poços mais profundos tendem a exigir mais tempo de operação devido à maior complexidade operacional e ao aumento do tempo de manobra das colunas.

---

## Objetivo

Validar essa hipótese utilizando técnicas de análise de dados em ambiente Big Data, aplicando:

- Modo Lazy com Polars
- Correlação de Pearson
- Regressão Linear Preditiva

In [ ]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# Convertendo .csv expandido para um arquivo .parquet
arquivo_csv_grande = "colar caminho do csv expandido"
arquivo_parquet_final = "colar nome do novo arquivo .parquet"  

## A exportação vai ser feita dentro de um try / except para evitar erros, visto que
## o arquivo a ser utilizado foi criado no final da primeira parte para atingir os crítérios
try:
    # 1. scan_csv() cria o plano (LazyFrame)
    lazy_conversao = pl.scan_csv(arquivo_csv_grande, separator=',', encoding='utf8-lossy')

    # 2. sink_parquet() executa em modo streaming e salva direto no disco
    lazy_conversao.sink_parquet(arquivo_parquet_final)
    
    print(f"Sucesso: Arquivo '{arquivo_csv_grande}' convertido para '{arquivo_parquet_final}'!\n")

except FileNotFoundError:
    print(f"ERRO CRÍTICO: O arquivo original '{arquivo_csv_grande}' não foi encontrado.")
    print("Garanta que o script do Pandas foi executado para gerar a base de Big Data.\n")
    sys.exit()  # O código é interrompido se o arquivo não existir

except Exception as e:
    print(f"ERRO INESPERADO NO PROCESSAMENTO: Ocorreu uma falha ao gravar o arquivo Parquet.")
    print(f"Detalhes técnicos do erro: {e}\n")
    sys.exit()  # O código é interrompido por erro inesperado

*Pipeline Lazy Mode e Processamento*


In [ ]:
# Criando o plano de execução Lazy a partir do arquivo .Parquet
inicial = (
    pl.scan_parquet(arquivo_parquet_final)  # ou caminho 'r"C:\Users\rafaella.almeida....'
    # Garante a definição correta do tipo de dado
    .with_columns([
        pl.col("profundidade_m").cast(pl.Float64),
        pl.col("Dias_em_intervencao").cast(pl.Int64)
    ])
    # Definindoo filtro de segurança para remover nulos
    .filter(
        (pl.col("Dias_em_intervencao").is_not_null()) & 
        (pl.col("profundidade_m").is_not_null())
    )
)

# Executando o plano Lazy para trazer o resumo estatístico
metricas_projeto = inicial.select([
    pl.col("profundidade_m").mean().alias("Média_Profundidade"),
    pl.col("Dias_em_intervencao").mean().alias("Média_Dias_Intervencao"),
    pl.col("Dias_em_intervencao").std().alias("Desvio_Padrao_Dias"),
    pl.col("Dias_em_intervencao").max().alias("Máximo_Dias_Intervencao")
]).collect()

print(f"Análise estatística\n{metricas_projeto}")

***Análise de Correlaão de Pearson***

*Avaliação do modelo (Cálculo das métricas estatísticas)*
*O Coeficiente de Determinação (R²) Mostra o quanto o modelo explica a realidade.*
*Ele Varia de 0 a 1 (0% a 100%). Quanto mais perto de 1, mais a profundidade explica os dias de intervenção.*
*Se o resultado for mais perto de 0, confirma que a profundidade não serve para 'prever' o tempo de sonda.*
*Fórmula: **"r2_modelo = r2_score(y, y_pred)"***

*O Erro Quadrático Médio (MSE) Mede o tamanho do erro do modelo.*
*Ele calcula a distância entre os pontos reais e a linha vermelha do gráfico, eleva ao quadrado e tira a média*.
*Quanto menor o valor (mais perto de zero), mais "certeiras" são as previsões da reta.*
*Fórmula: "**mse_modelo = mean_squared_error(y, y_pred)"***

In [ ]:
# Coletando apenas as colunas necessárias para consulta
df_coletado = inicial.select(["profundidade_m", "Dias_em_intervencao"]).collect()

correlacao = df_coletado.select(
    pl.corr("profundidade_m", "Dias_em_intervencao", method="pearson").alias("r")
)

*A seguir, o r (Pearson): Mede a força e a direção da relação. Se der um número positivo perto de 1,a linha sobe; se der negativo perto de -1, a linha desce; se der perto de 0, vira uma espécie de "nuvem".*

*O r2 (rxx2): Traduz essa relação em porcentagem. Se o r de Pearson for, por exemplo, 0.20 (uma correlação fraca), o seu r2 será '0.20x0.20 = 0.04'.*

*Isso significaria que a profundidade explica apenas 4% dos dias que o poço fica em intervenção.*
*Logo, os outros 96% dos dias são devido a outras variáveis.*

In [ ]:
r = correlacao["r"][0]
r2 = r**2

print(f"Coeficiente de Pearson (r): {r:.4f}")
print(f"Coeficiente de Determinação (r²): {r2:.4f}")

In [ ]:
# Grau de dispersão dos pontos
if abs(r) < 0.3:
    print("Força: Correlação FRACA. Nuvem de pontos espalhada e sem uma forma linear definida.")
    print("Conclusão: A profundidade do poço não dita diretamente a duração da intervenção.")
elif abs(r) > 0.7:
    print("Força: Correlação FORTE. Os pontos seguem uma tendência linear bem definida.")
    print("Conclusão: A profundidade é um fator determinante para os dias de sonda.")
else:
    print("Força: Correlação MODERADA. Existe uma tendência, mas com dispersão notável.")

In [ ]:
# Comportamento da Linha (direção e inclinação)
if abs(r) < 0.1: # Se for quase zero, a correlação é nula
    print("Direção: Correlação NULA ou NEUTRA. A linha de tendência é praticamente horizontal.")
elif r > 0:
    print("Direção: Correlação POSITIVA (r > 0). Linha subindo da esquerda para a direita (/).")
    print("Significado: À medida que a profundidade aumenta, o tempo de intervenção também tende a aumentar.")
else:
    print("Direção: Correlação NEGATIVA (r < 0). Linha descendo da esquerda para a direita (\).")
    print("Significado: À medida que a profundidade aumenta, o tempo de intervenção tende a diminuir.")

*Desenhando o gráfico...*

In [ ]:
# Gráfico de Dispersão
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_coletado.to_pandas(),
    x="profundidade_m",
    y="Dias_em_intervencao",
    alpha=0.3,
    color="blue"
)
plt.title(f'Dispersão Big Data: Profundidade vs Dias em Intervenção (r={r:.2f})')
plt.xlabel('Profundidade do Poço (Metros)')
plt.ylabel('Dias em Intervenção')
plt.grid(True, alpha=0.3)
# plt.xscale('log') # Escala logarítmica, validar necessidade de usar no código
plt.show()

*Regressão Linear - Análise Preditiva*

In [ ]:
## 1. Preparar os dados para o modelo
# Converter para arrays numpy

X = df_coletado["profundidade_m"].to_numpy().reshape(-1, 1) # Variável independente (profundidade em metros)
y = df_coletado["Dias_em_intervencao"].to_numpy()   # Variável independente (dias em intervencao)

## 2. Criar e treinar o modelo
modelo = LinearRegression()
modelo.fit(X, y)

# Coeficientes do modelo
inclinacao = modelo.coef_[0]
intercepto = modelo.intercept_

print(f"Equação da reta: DIAS_EM_INTERVENCAO = {inclinacao:.4f} * PROFUNDIDADE + {intercepto:.2f}")

In [ ]:
## 3. Fazer previsões
y_pred = modelo.predict(X)

## 4. Avaliar o modelo
# (from sklearn.metrics import r2_score, mean_squared_error)

r2_modelo = r2_score(y, y_pred)
mse_modelo = mean_squared_error(y, y_pred)

print(f"Métricas de avaliação:")
print(f"-> Coeficiente de Determinação (R²): {r2_modelo:.4f}")
print(f"-> Erro Quadrático Médio (MSE): {mse_modelo:.2f}")

*Desenhando o gráfico...*

In [ ]:
## 5. Visualização dos resultados
plt.figure(figsize=(10, 6))

# Plotar dados reais
plt.scatter(X, y, color='gray', alpha=0.3, label='Dados Reais (Poços)')

# Plotar linha de regressão
plt.plot(X, y_pred, color='red', linewidth=2, label=f'Regressão: y = {inclinacao:.4f}x + {intercepto:.2f}')

# Configurações do gráfico
plt.title('Análise Preditiva: Regressão Linear (Profundidade vs Intervenção)')
plt.xlabel('Profundidade do Poço (Metros)')
plt.ylabel('Dias em Intervenção')
plt.grid(True, alpha=0.3)
plt.legend()

# Mostrar gráfico
plt.tight_layout()
plt.show()

## ***PREPARAÇÃO E EXPORTAÇÃO PARA O POWER BI***

In [ ]:
# 1. Calcular os limites de quartis (Q1 e Q3) baseados na profundidade dos poços
q1_profund = df_coletado.select(pl.col("profundidade_m").quantile(0.25)).item()
q3_profund = df_coletado.select(pl.col("profundidade_m").quantile(0.75)).item()

print(f"Poços abaixo de {q1_profund:.2f}m serão classificados como 'Poço Raso'")
print(f"Poços acima de {q3_profund:.2f}m serão classificados como 'Poço Profundo'")

In [ ]:
# 2. Criar a coluna "Etiqueta" de classificação por faixa de profundidade
# Isso enriquece os filtros e segmentadores dentro do Power BI
df_export_pbi = df_coletado.with_columns(
    pl.when(pl.col("profundidade_m") > q3_profund).then(pl.lit("Poço Profundo"))
    .when(pl.col("profundidade_m") < q1_profund).then(pl.lit("Poço Raso"))
    .otherwise(pl.lit("Poço Moderado")) # Apenas para preencher, mas vamos filtrar
    .alias("Etiqueta_Profundidade")
    ).filter(
    (pl.col("Etiqueta_Profundidade") == "Poço Profundo") | 
    (pl.col("Etiqueta_Profundidade") == "Poço Raso")
)

# 3. Exportar para CSV (Simulando a entrega para o time de Dataviz)
nome_csv_pbi = "intervencao_pocos_pbi.csv"
df_export_pbi.write_csv(nome_csv_pbi)

print(f"Sucesso! Arquivo '{nome_csv_pbi}' gerado com sucesso.")
print(df_export_pbi.head())

**Criando medidas básicas usando DAX**          
    *As medidas deverão ser replicadas dentro do PowerBI*

In [ ]:
# Medida Media_Dias_Intervencao: Calcula a média geral de dias que os poços ficam parados em intervenção
Media_Dias_Intervencao = AVERAGE('intervencao_pocos_pbi'[Dias_em_intervencao])

In [ ]:
# Medida Media_Dias_Pocos_Profundos: Calcula a média de dias filtrando os os poços mais profundos
Media_Dias_Pocos_Profundos = 
CALCULATE(
    [Media_Dias_Intervencao],
    'intervencao_pocos_pbi'[Etiqueta_Profundidade] = "Poço Profundo"
)

In [ ]:
# Medida Total_Pocos_Analisados: Faz a contagem total de poços na base de Big Data
Total_Pocos_Analisados = COUNTROWS('intervencao_pocos_pbi')